<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

## Classification problem on molecular graphs using Graph Neural Networks (GNN) and `pytorch-geometric`

**Goal** Familiarize `pytorch-geometric` in handling GNNs and `DataLoaders`, and classify whether a molecule is toxic or not (**molecular level binary-property**)

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Training dataset** : goal is to predict chemical toxicity using Graph Neural Networks

* ~7,800 molecules represented by SMILES strings, each with the outputs from 12 binary classification assays. Labels are either 1 = active, 0 = inactive or NaN = not tested

* Assays include nuclear receptor signaling pathways (7 assays) and stress response pathways (5 assays)
</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Tools**
* `scikit-learn`, `torch`
* **Core cheminfo**: `RDKit` **fingerprints, descriptors** to automate molecular representation and generate the input to the classifier
* `pytorch-geometric` to featurize the molecualr graphs, define the GNN layers, the head of network is a classifier
* `torch` to train and test, using **batches** and **early_stopping**
</div>

In [ ]:
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.1+cu118.html
!pip install torch-geometric

!pip install rdkit-pypi

Looking in links: https://data.pyg.org/whl/torch-2.0.1+cu118.html


In [ ]:
import pandas as pd
import numpy as np
import torch, os, joblib, sys
import torch.nn as nn
import torch.nn.functional as F    # activation functions

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


from torch.utils.data import random_split
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, GraphConv
import copy

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, AllChem

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib
%matplotlib inline

import matplotlib.pyplot as plt

from utils import mol_to_nx, visualize_molecular_graph, atom_features, bond_features, graph_featurizer_pygeom
from utils import split_data, model_testing_GNN

In [ ]:
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, GraphConv
import copy

from utils import model_testing_GNN

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### Read in **Tox21** dataset, pick one classification task and featurize (calculate molecular graph embeddings as `geometric` Data objects)
</div>

In [ ]:
# Tox21 da MoleculeNet, downloaded from the internet, sicne the original link/url comes with restrictions
df = pd.read_csv('tox21.csv')

# Print columns
print(list(df.columns))  # print first few databse columns (SMILES + target)

print('Number of molecules in dataset = ' + str(df.shape[0]))
print('Number of assays available for classification tasks = ' + str(df.shape[1]-2))

['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'mol_id', 'smiles']
Number of molecules in dataset = 7831
Number of assays available for classification tasks = 12


In [ ]:
task = 'NR-AR'

print(f'Classification task to choose = {task}')
if task not in list(df.columns):
    print('acthung! there is no such classification task in the input file')

Classification task to choose = NR-AR


In [ ]:
def graph_featurizer_pygeom(mol, mol_id, y, edge_attrib = None):

    """A molecule is translated into a featurized graphs, with nodes and node labels + edges and edge labels (if applicable) """

    atom_feats = []
    edge_index = []
    edge_attr = []

    for atom in mol.GetAtoms():
        atom_feats.append(atom_features(atom))     # list of torch tensors

    for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()
            edge_index.append([i, j])
            edge_index.append([j, i])  # graph is undirected

            if edge_attrib is not None:
                edge_attr.append(bond_features(bond))
                edge_attr.append(bond_features(bond)) # if bidirectional, we miss half the bonds if we don't include the flipped one
                                                  # clearly here [i,j] and [j,i] share the same feature
    # from list of torch tensors to one torch tensor
    x = torch.stack(atom_feats)

    if edge_attrib is not None:
        edge_attr = torch.stack(edge_attr)

    # from a list of numpy arrays to a torch tensor
    edge_index = torch.tensor(edge_index, dtype=torch.long).T   # now we need to transpose this for compatibility sake

    #print(x.shape)    # (number of atoms, number of features)
    #print(edge_index.shape)    # (2, number of edges * 2), each edge is listed twice (i,j and j,i)
    #print(edge_attr.shape)    # (number of edges * 2, size of one-hot encoding for edge embeddings, if applicable

    if edge_attrib is not None:
        data = Data(x=x, edge_index=edge_index, edge_attr = edge_attr, y=y)     # this is a torch-geometric dataset format, compatible with NNs syntax
    else:
        data = Data(x=x, edge_index=edge_index, y=y)
    data.idx = torch.tensor([mol_id])     # keep track of mol_id, since later shuffling

    return data

In [ ]:
def split_data(full_data, train_ratio, val_ratio):

    """
    Helper function, split dataset into train, validation and test set, and define batches

    INPUT: * full_data, torch Data object, featurization (and output) for all molecular graphs
           * train_ratio (float): percentage of all data to allocate for training
           * val_ratio (float): percentage of all data to allocate for validation

    OUTPUT: * train_dataset (pytorch  Data object): training dataset
            * val_dataset (pytorch  Data object): validation dataset
            * test_dataset (pytorch Data object): test dataset
    """

    total_size = len(full_data)
    train_size = int(train_ratio * total_size)
    val_size   = int(val_ratio * total_size)
    test_size  = total_size - train_size - val_size  # handles rounding

    train_dataset, test_dataset, val_dataset = random_split(full_data, [train_size, test_size, val_size])

    return train_dataset, val_dataset, test_dataset

In [ ]:
full_data = []

for k, smile in enumerate(df['smiles'].values):

  # check if that task label is present or not (some assays were inconclusive on some molecules)
  if np.isnan(df[task].values[k]) == False:
    full_data.append(graph_featurizer_pygeom(Chem.MolFromSmiles(smile), k, df[task].values[k], edge_attrib=None))

print(len(full_data))

[18:53:32] WARNING: not removing hydrogen atom without neighbors
/tmp/ipython-input-255629490.py:29: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3725.)
  edge_index = torch.tensor(edge_index, dtype=torch.long).T   # now we need to transpose this for compatibility sake


7265


<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### Define train, test and validation datasets, all organized in batches

</div>

In [ ]:
# split into train, test, validation set using pytorch functionalities
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

train_dataset, val_dataset, test_dataset = split_data(full_data, train_ratio, val_ratio)

In [ ]:
n_samples = len(full_data)
print(f'Total nubmber of samples = {n_samples}')

in_dim = train_dataset[0].x.shape[1]
print('Embedding size of nodes = ' + str(in_dim))

print(f'Number of training samples = {len(train_dataset)}')

Total nubmber of samples = 7265
Embedding size of nodes = 5
Number of training samples = 5085


In [ ]:
# convert to torch-geometric Datasets, using batches
n_batches = 25

train_loader = DataLoader(train_dataset, batch_size=n_batches, shuffle=True) # batch size to be used, shuffle ON for training
test_loader  = DataLoader( test_dataset, batch_size=n_batches, shuffle=False)
val_loader   = DataLoader(  val_dataset, batch_size=n_batches, shuffle=False)

/usr/local/lib/python3.11/dist-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Define the GNN Classifier for a graph-based classification; use layers available in `torch-geometric`
        `GraphConv` + '`Relu` + `GraphConv` + `relu` + `global pooling` + `classifier`
    
Compare with hard-coded GNN tools
</div>

In [ ]:
class GNNclassifier(nn.Module):

    def __init__(self, in_dim, hidden_dim):
        super().__init__()

        """
        in_dim (int): size of input node embeddings
        hidden_dim (int): size of node embeddings after passing throuhg first laye
        """

        self.conv1 = GraphConv(in_dim, hidden_dim)     # this is aready a GNN layer that performs message passing with the nearest neighobrs
        self.conv2 = GraphConv(hidden_dim, hidden_dim)
        self.classifier = torch.nn.Linear(hidden_dim, 1)  # single Linear, we can always make this more complex

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)    # the linear layer is already implemented
        x = F.relu(x) # just need to apply a non linear activation function
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)  # graph embedding, averaging over all the noves
        x = self.classifier(x)     # logits per graph, shape [batch_size, 1]
        return x

In [ ]:
def model_training_classifier(model, optimizer, loss_fn, train_loader, val_loader, n_epochs, device, patience, early_stop = None):

    """
    Train
    """

    best_val_loss = float('inf')
    best_model_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    model.to(device)

    for epoch in range(n_epochs):

        model.train()
        total_train_loss = 0

        # train model
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            # extract features ('X') and labels ('y')
            pred = model(batch.x, batch.edge_index, batch.batch)
            labels = batch.y.float().unsqueeze(1)

            # compute the loss function and backpropagate using the user-specified optimizer
            loss = loss_fn(pred, labels)
            loss.backward()
            optimizer.step()

            # add loss from this batch to the total training loss
            total_train_loss += loss.item()

        if epoch % 10 == 0 or epoch == n_epochs - 1:

            # model optimized for now, ready to propagate forward
            model.eval()
            with torch.no_grad():

                total_correct = 0
                total = 0
                total_val_loss = 0

                for val_batch in val_loader:

                    val_batch = val_batch.to(device)

                    # extract the input and output for the batch, compute the loss for the batch, add that to the full validation loss
                    pred = torch.sigmoid(model(val_batch.x, val_batch.edge_index, val_batch.batch)).squeeze(1)
                    labels = val_batch.y.view(-1)

                    val_loss = loss_fn(pred, labels)
                    total_val_loss += val_loss.item()

                    # classify: map probabilities to binary 0 and 1
                    predicted = (pred > 0.5).float().view(-1)
                    total += labels.size(0)

                    total_correct += (predicted == labels).sum().item() # count those predictions that match the known labels

                # average loss on the validation dataset
                avg_val_loss = total_val_loss / len(val_loader)
                val_acc = total_correct / total

                print(f"Epoch {epoch} | Train Loss: {total_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}")

            # === Early stopping check ===
            if early_stop is not None:

               if avg_val_loss < best_val_loss:     # if performance on validation set is still improving, keep going
                  best_val_loss = avg_val_loss
                  best_model_state = copy.deepcopy(model.state_dict())    # save model parameters
                  patience_counter = 0
               else:
                  patience_counter += 1
                  if patience_counter >= patience:    # performance on validation set stopped improving
                      print(f"Early stopping at epoch {epoch}. Best val loss: {best_val_loss:.4f}")
                      break

    # Load best weights
    model.load_state_dict(best_model_state)

    return model

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Instantiate model, optimizer and Binary Cross Entropy (w Logits) function; then train the model using early stopping
Please note the model we chose (GNVConv), cannot handle edge message passing
</div>

In [ ]:
# instantiate model
model = GNNclassifier(in_dim = in_dim, hidden_dim=8).to(device)  # in_dim is hard_coded, we might automate this based on the features
print(model)  # print the architecture in terms of layers and input/output sizes

# define optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.BCEWithLogitsLoss()

n_epochs = 200
patience = 10

model = model_training_classifier(model, optimizer, loss_fn, train_loader, val_loader, n_epochs, device, patience, early_stop = 'on')

GNNclassifier(
  (conv1): GraphConv(5, 8)
  (conv2): GraphConv(8, 8)
  (classifier): Linear(in_features=8, out_features=1, bias=True)
)
Epoch 0 | Train Loss: 50.5507 | Val Loss: 0.7131 | Val Acc: 0.96
Epoch 10 | Train Loss: 31.5369 | Val Loss: 0.7064 | Val Acc: 0.96
Epoch 20 | Train Loss: 27.5674 | Val Loss: 0.7033 | Val Acc: 0.97
Epoch 30 | Train Loss: 25.9872 | Val Loss: 0.7063 | Val Acc: 0.97
Epoch 40 | Train Loss: 25.6289 | Val Loss: 0.7028 | Val Acc: 0.97
Epoch 50 | Train Loss: 25.4954 | Val Loss: 0.7019 | Val Acc: 0.97
Epoch 60 | Train Loss: 24.9132 | Val Loss: 0.7037 | Val Acc: 0.97
Epoch 70 | Train Loss: 24.6357 | Val Loss: 0.7072 | Val Acc: 0.97
Epoch 80 | Train Loss: 24.3175 | Val Loss: 0.7017 | Val Acc: 0.97
Epoch 90 | Train Loss: 24.7730 | Val Loss: 0.7008 | Val Acc: 0.97
Epoch 100 | Train Loss: 24.3368 | Val Loss: 0.7016 | Val Acc: 0.97
Epoch 110 | Train Loss: 25.0995 | Val Loss: 0.7084 | Val Acc: 0.97
Epoch 120 | Train Loss: 24.5448 | Val Loss: 0.7011 | Val Acc: 0.97
Epoc

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Test trained GNN optimizer on the test set molecular graphs, compute metrics
</div>

In [ ]:
# Set model in evaluation mode
acc, pred, red, f1, auc = model_testing_GNN(model, test_loader)

NameError: name 'accuracy_score' is not defined